<a href="https://colab.research.google.com/github/HYUNSUNG03/AI_study/blob/main/Personal_Information_Detecting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers datasets seqeval torch easyocr opencv-python-headless -q

In [4]:
# 코랩 상단 메뉴: 수정 → 노트북 설정 → 위젯 출력 지우기
# 또는 아래 코드 실행
from IPython.display import clear_output
clear_output()

In [ ]:
import torch
import numpy as np
import pandas as pd
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
from datasets import Dataset
import seqeval
from seqeval.metrics import f1_score, precision_score, recall_score, classification_report

print(f"GPU 사용 가능: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU 이름: {torch.cuda.get_device_name(0)}")

In [ ]:
# BIO 태그 방식
# B = Begin (엔티티 시작)
# I = Inside (엔티티 중간)
# O = Outside (개인정보 아님)

LABEL_LIST = [
    "O",
    "B-PER",   # 이름 시작
    "I-PER",   # 이름 계속
    "B-LOC",   # 주소 시작
    "I-LOC",   # 주소 계속
    "B-ORG",   # 기관명 시작
    "I-ORG",   # 기관명 계속
    "B-PHONE", # 전화번호 시작
    "I-PHONE", # 전화번호 계속
    "B-EMAIL", # 이메일 시작
    "I-EMAIL", # 이메일 계속
    "B-NUM",   # 주민번호/계좌번호 시작
    "I-NUM",   # 주민번호/계좌번호 계속
]

LABEL2ID = {label: i for i, label in enumerate(LABEL_LIST)}
ID2LABEL = {i: label for i, label in enumerate(LABEL_LIST)}

print(f"총 레이블 수: {len(LABEL_LIST)}")
print(f"레이블 목록: {LABEL_LIST}")

In [ ]:
# 직접 만드는 합성 데이터셋
# 형식: (문장, [토큰 리스트], [레이블 리스트])
# 실제 프로젝트에서는 이 데이터를 더 많이 추가할수록 성능이 올라감

raw_data = [
    {
        "tokens": ["저는", "김민준", "입니다", "."],
        "labels": ["O", "B-PER", "O", "O"]
    },
    {
        "tokens": ["제", "이름은", "이수진", "이고", "나이는", "28살", "입니다", "."],
        "labels": ["O", "O", "B-PER", "O", "O", "O", "O", "O"]
    },
    {
        "tokens": ["박철수", "씨의", "연락처는", "010-1234-5678", "입니다", "."],
        "labels": ["B-PER", "O", "O", "B-PHONE", "O", "O"]
    },
    {
        "tokens": ["전화번호", ":", "010-9876-5432", "로", "연락", "주세요", "."],
        "labels": ["O", "O", "B-PHONE", "O", "O", "O", "O"]
    },
    {
        "tokens": ["이메일은", "hong.gildong@naver.com", "입니다", "."],
        "labels": ["O", "B-EMAIL", "O", "O"]
    },
    {
        "tokens": ["문의", "사항은", "support@company.co.kr", "로", "보내주세요", "."],
        "labels": ["O", "O", "B-EMAIL", "O", "O", "O"]
    },
    {
        "tokens": ["주소는", "서울특별시", "강남구", "테헤란로", "123", "입니다", "."],
        "labels": ["O", "B-LOC", "I-LOC", "I-LOC", "I-LOC", "O", "O"]
    },
    {
        "tokens": ["부산시", "해운대구", "센텀시티로", "99에", "살고", "있어요", "."],
        "labels": ["B-LOC", "I-LOC", "I-LOC", "I-LOC", "O", "O", "O"]
    },
    {
        "tokens": ["주민번호는", "901215-1234567", "입니다", "."],
        "labels": ["O", "B-NUM", "O", "O"]
    },
    {
        "tokens": ["계좌번호", ":", "123-456-789012", "로", "입금해주세요", "."],
        "labels": ["O", "O", "B-NUM", "O", "O", "O"]
    },
    {
        "tokens": ["삼성전자", "김영호", "부장님께", "메일", "보내주세요", "."],
        "labels": ["B-ORG", "B-PER", "O", "O", "O", "O"]
    },
    {
        "tokens": ["저는", "카카오", "다니는", "정하늘", "입니다", "."],
        "labels": ["O", "B-ORG", "O", "B-PER", "O", "O"]
    },
    {
        "tokens": ["연락처는", "02-1234-5678", "이고", "담당자는", "최지원", "입니다", "."],
        "labels": ["O", "B-PHONE", "O", "O", "B-PER", "O", "O"]
    },
    {
        "tokens": ["제", "이메일", "kim.minji@kakao.com", "과", "전화", "010-5555-6666", "입니다", "."],
        "labels": ["O", "O", "B-EMAIL", "O", "O", "B-PHONE", "O", "O"]
    },
    {
        "tokens": ["경기도", "성남시", "분당구", "판교로", "235에", "위치한", "네이버", "본사", "."],
        "labels": ["B-LOC", "I-LOC", "I-LOC", "I-LOC", "I-LOC", "O", "B-ORG", "O", "O"]
    },
    {
        "tokens": ["이름", ":", "홍길동", ",", "생년월일", ":", "1985-03-22"],
        "labels": ["O", "O", "B-PER", "O", "O", "O", "B-NUM"]
    },
    {
        "tokens": ["오늘", "날씨가", "정말", "좋네요", "."],
        "labels": ["O", "O", "O", "O", "O"]
    },
    {
        "tokens": ["이번", "회의는", "3시에", "시작합니다", "."],
        "labels": ["O", "O", "O", "O", "O"]
    },
    {
        "tokens": ["제품", "문의는", "고객센터로", "연락해주세요", "."],
        "labels": ["O", "O", "O", "O", "O"]
    },
    {
        "tokens": ["오윤서", "씨", "주민번호", "850712-2345678", "확인", "부탁드립니다", "."],
        "labels": ["B-PER", "O", "O", "B-NUM", "O", "O", "O"]
    },
]

print(f"총 학습 샘플 수: {len(raw_data)}")
print(f"\n샘플 예시:")
for i, d in enumerate(raw_data[:2]):
    print(f"  [{i}] 토큰: {d['tokens']}")
    print(f"       레이블: {d['labels']}")

In [ ]:
MODEL_NAME = "monologg/koelectra-base-v3-discriminator"
# 파인튜닝 안 된 순수 기반 모델 사용 (우리가 직접 NER 학습)

print("⏳ 토크나이저 로딩 중...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("✅ 토크나이저 로드 완료!")


def tokenize_and_align_labels(tokens, labels, max_length=128):
    """
    토크나이저가 단어를 서브워드로 쪼갤 때 레이블을 맞춰주는 함수
    예: '김민준' → ['김', '##민', '##준'] 이 될 때
        레이블:  B-PER → [B-PER, -100, -100]
        (-100은 loss 계산에서 무시됨)
    """
    tokenized = tokenizer(
        tokens,
        is_split_into_words=True,  # 이미 토큰 분리된 입력
        max_length=max_length,
        padding="max_length",
        truncation=True,
        return_tensors=None
    )

    word_ids = tokenized.word_ids()
    aligned_labels = []
    prev_word_id = None

    for word_id in word_ids:
        if word_id is None:
            # [CLS], [SEP], [PAD] 토큰 → -100
            aligned_labels.append(-100)
        elif word_id != prev_word_id:
            # 단어의 첫 서브워드 → 원래 레이블 사용
            aligned_labels.append(LABEL2ID[labels[word_id]])
        else:
            # 단어의 이후 서브워드 → -100 (loss 무시)
            aligned_labels.append(-100)
        prev_word_id = word_id

    tokenized["labels"] = aligned_labels
    return tokenized


# 전체 데이터 전처리
processed_data = []
for item in raw_data:
    result = tokenize_and_align_labels(item["tokens"], item["labels"])
    processed_data.append(result)

# train / validation 분리 (8:2)
split_idx = int(len(processed_data) * 0.8)
train_data = processed_data[:split_idx]
val_data = processed_data[split_idx:]

# HuggingFace Dataset 형식으로 변환
def to_dataset(data):
    return Dataset.from_dict({
        "input_ids":      [d["input_ids"] for d in data],
        "attention_mask": [d["attention_mask"] for d in data],
        "token_type_ids": [d.get("token_type_ids", [0]*len(d["input_ids"])) for d in data],
        "labels":         [d["labels"] for d in data],
    })

train_dataset = to_dataset(train_data)
val_dataset   = to_dataset(val_data)

print(f"✅ 데이터 전처리 완료!")
print(f"   학습 데이터: {len(train_dataset)}개")
print(f"   검증 데이터: {len(val_dataset)}개")

In [ ]:
print("⏳ KoELECTRA 모델 로딩 중...")

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL_LIST),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True  # 분류 헤드 새로 붙이기
)

print(f"✅ 모델 로드 완료!")
print(f"   파라미터 수: {sum(p.numel() for p in model.parameters()):,}")
print(f"   학습 가능 파라미터: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
def compute_metrics(eval_pred):
    """
    seqeval 기반 NER 평가 지표
    Precision, Recall, F1 계산
    """
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=2)

    true_labels = []
    true_preds  = []

    for pred_seq, label_seq in zip(predictions, labels):
        true_label_row = []
        true_pred_row  = []
        for pred, label in zip(pred_seq, label_seq):
            if label == -100:
                continue  # 패딩, 서브워드 무시
            true_label_row.append(ID2LABEL[label])
            true_pred_row.append(ID2LABEL[pred])
        true_labels.append(true_label_row)
        true_preds.append(true_pred_row)

    return {
        "precision": precision_score(true_labels, true_preds),
        "recall":    recall_score(true_labels, true_preds),
        "f1":        f1_score(true_labels, true_preds),
    }

print("✅ 평가 지표 정의 완료!")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

OUTPUT_DIR = "/content/drive/MyDrive/privacy_ner_model"

data_collator = DataCollatorForTokenClassification(tokenizer)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=10,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=3e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=10,
    report_to="none",
    fp16=torch.cuda.is_available()
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,    # tokenizer → processing_class 로 변경
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("🚀 파인튜닝 시작!")
trainer.train()
print("✅ 학습 완료!")